In [ ]:
from dotenv import load_dotenv  # loads variables from .env file into os.environ
import os                        # lets us read environment variables

load_dotenv()                    # reads .env in the current working directory
api_key = os.getenv("ANTHROPIC_API_KEY")  # retrieves the key (None if missing)
print("API key loaded:", api_key is not None)  # True = key found, False = check .env

In [ ]:
from anthropic import Anthropic  # the official Anthropic Python SDK

client = Anthropic(api_key=api_key)  # creates a reusable client bound to our key
print("Anthropic client ready")       # confirms no import or auth error so far

In [ ]:
response = client.messages.create(
    model="claude-haiku-4-5",   # fast, cheap model — ideal for smoke tests
    max_tokens=50,               # hard ceiling on output length
    messages=[{"role": "user", "content": "Say hello in one short sentence."}]
)
print(response.content[0].text)  # content is a list; [0] is the first (only) block

# CCA Lab: Prompt Engineering — System Prompts & Error Handling

**Course:** Anthropic Academy · **Sub-module:** Prompt Engineering · **Lesson:** System Prompts & Failure Modes

## What this lab covers
- How the `system` parameter shapes model behaviour at an architectural level
- Decision framework: what belongs in `system` vs the `user` turn
- Live behavioural comparison: with vs without a system prompt
- Multi-turn conversation with messages list accumulation and token cost analysis
- Common failure modes, their triggers, root causes, and how to catch them in code
- Token usage tracking across every API call in the session
- Improvement cycle: write → evaluate → improve → re-evaluate (3 versions)

## CCA Domains Exercised
`Prompt Engineering` · `Agentic Architecture` (error handling)

## Learning Objectives
1. Explain what the `system` parameter does and when to use it.
2. Distinguish `system`-appropriate content from `user`-turn content.
3. Deliberately trigger and catch API errors caused by bad parameter usage.
4. Build and observe a multi-turn conversation via messages list accumulation.
5. Apply a 3-version improvement cycle with a numeric rubric and threshold.
6. Track per-call and cumulative token usage across a session.

## CCA objective demonstrated: Environment hygiene and parameter literacy

## Section 1: Prerequisites & API Glossary

| Parameter | Type | Required | Description |
|-----------|------|----------|-------------|
| `model` | `str` | ✅ | Model ID string, e.g. `claude-haiku-4-5` |
| `max_tokens` | `int` | ✅ | Hard ceiling on output tokens |
| `messages` | `list[dict]` | ✅ | Alternating `user`/`assistant` turns |
| `system` | `str` | ❌ | Persistent persona/instructions prepended to every turn |
| `temperature` | `float` | ❌ | Randomness 0–1 (default 1) |
| `stop_sequences` | `list[str]` | ❌ | Strings that halt generation early |

**Install check:** `pip install anthropic python-dotenv`

## CCA objective demonstrated: Reusable API wrapper with usage tracking

## Section 2: The `ask()` Helper

In [ ]:
usage_log = []  # session-wide list; every API call appends one entry here

def ask(prompt, system=None, model="claude-haiku-4-5", max_tokens=256):
    """
    Send a single-turn message and return the assistant's reply text.

    Parameters
    ----------
    prompt     : str   — the user message
    system     : str | None — optional system prompt (omitted if None)
    model      : str   — Anthropic model ID
    max_tokens : int   — hard output-token ceiling

    Returns
    -------
    str — the assistant's reply
    """
    # Build the base kwargs dict with always-required parameters
    kwargs = {
        "model": model,           # which Claude model to call
        "max_tokens": max_tokens,  # prevents runaway output; required by API
        "messages": [{"role": "user", "content": prompt}],  # messages must be a list of dicts
    }
    # Only add 'system' key when we actually have a non-None string.
    # Passing system=None explicitly raises a validation error because the
    # API schema requires system to be a string when the key is present.
    if system is not None:          # guard: omit the key entirely if no system prompt
        kwargs["system"] = system   # string value → key is included safely

    # Unpack the dict as keyword arguments — equivalent to writing each param by name
    response = client.messages.create(**kwargs)

    # response.content is a LIST of content blocks (text, tool_use, image, etc.)
    # For a plain text response there is always exactly one block, so [0] is safe.
    # We access .text on that block to get the string.
    reply = response.content[0].text

    # response.stop_reason explains WHY generation stopped:
    #   'end_turn'   — model chose to stop naturally (normal)
    #   'max_tokens' — hit our ceiling (response may be TRUNCATED — check this!)
    stop = response.stop_reason

    # Append a record to usage_log so we can analyse token consumption later.
    # We truncate prompt/system for display; full values are in the API response.
    usage_log.append({
        "call": len(usage_log) + 1,              # sequential call number
        "system": (system or "")[:40],           # first 40 chars for display
        "prompt": prompt[:40],                   # first 40 chars for display
        "input_tokens": response.usage.input_tokens,   # tokens sent to the model
        "output_tokens": response.usage.output_tokens, # tokens the model generated
        "stop_reason": stop,                     # why generation ended
    })
    return reply  # caller gets the plain text string

print("ask() helper defined — usage_log initialised")

## CCA objective demonstrated: Agentic Architecture — catching and recovering from API validation errors

## Section 3: D-ERR — Intentional Error Demonstration

Passing `system=None` directly as a keyword argument (rather than omitting the key)
causes the Anthropic SDK to raise an error because the API schema requires `system`
to be a **string** when present — `None` is not a valid string.  
Below we deliberately trigger this, catch it, and show the corrected conditional pattern.

In [ ]:
import anthropic  # needed to reference anthropic exception classes

# ── DELIBERATELY BROKEN CALL ──────────────────────────────────────────────────
print("=== Triggering error: system=None passed explicitly ===")
try:
    bad_response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=50,
        system=None,   # ← BUG: SDK/API rejects None; key must be absent or a string
        messages=[{"role": "user", "content": "Hello"}],
    )
except Exception as e:
    # Catch broadly so we can print the actual error class name for learning purposes
    print(f"ERROR TYPE : {type(e).__name__}")
    print(f"ERROR MSG  : {e}")
    print()

# ── CORRECTED PATTERN ─────────────────────────────────────────────────────────
print("=== Corrected: conditionally omit system key when None ===")

def safe_create(prompt, system=None):
    """Build kwargs dict and only include 'system' when it has a real string value."""
    kwargs = {
        "model": "claude-haiku-4-5",
        "max_tokens": 50,
        "messages": [{"role": "user", "content": prompt}],
    }
    if system is not None:       # guard: only add key when value is a meaningful string
        kwargs["system"] = system  # key is present only when value is safe
    return client.messages.create(**kwargs)

ok = safe_create("Hello", system=None)   # system=None → key is omitted entirely
print("Corrected call succeeded:", ok.content[0].text[:80])

## CCA objective demonstrated: Prompt Engineering — system parameter architecture and behavioural divergence

## Section 4: D-SYS — System Prompt: What It Is, When to Use It, Why It Matters

The `system` parameter is a **privileged instruction block** that sits above the
conversation history. Unlike user messages, it cannot be "forgotten" by later turns
and is not attributed to either speaker — it represents the *developer's voice*.

**The API prepends the system block to the conversation before the first user turn;
it cannot be overridden or forgotten by subsequent messages.**

### Decision Table: system vs user turn

| Put this in `system` | Put this in the `user` turn |
|----------------------|-----------------------------|
| Persona / role definition | The actual question or task |
| Output format rules (always JSON, always bullet points) | Dynamic data or context that changes per request |
| Domain restrictions (only answer maths questions) | Examples specific to one interaction |
| Tone & language instructions | User-provided files or retrieved documents |
| Safety guardrails that must persist | Follow-up clarifications |

**Architectural principle:** Place anything that must remain constant and invisible
to the end-user in `system`; place anything dynamic, user-supplied, or
request-specific in the `user` turn.

Below we run the **same user message** with and without a math-tutor system prompt
to observe behavioural divergence.

In [ ]:
MATH_TUTOR_SYSTEM = """You are a patient maths tutor for 10-year-old students.
Always explain your reasoning step by step using simple language.
End every response with an encouraging sentence."""

USER_QUESTION = "What is 7 multiplied by 8, and why?"

# Call WITHOUT system prompt — default Claude behaviour, no persona constraints
reply_no_sys = ask(USER_QUESTION, system=None)

# Call WITH math-tutor system prompt — constrained persona, tone, and structure
reply_with_sys = ask(USER_QUESTION, system=MATH_TUTOR_SYSTEM)

print("═" * 60)
print("WITHOUT system prompt:")
print("─" * 60)
print(reply_no_sys)
print()
print("═" * 60)
print("WITH math-tutor system prompt:")
print("─" * 60)
print(reply_with_sys)
print()
print("Observation: the system prompt imposes tone, structure, and a closing")
print("encouragement that is absent in the unconstrained response.")

## CCA objective demonstrated: Operationalising quality with a numeric rubric — per-criterion scoring with pass/fail icons

## Section 5: Core Concept — Rubric Scoring

A good rubric converts subjective quality into a reproducible number.
Each dimension emits either 0 or its max score, printed with a ✅/❌ icon.
The `PASS_THRESHOLD` constant lives adjacent to the function so any reader
can see the scoring bar without searching elsewhere.

In [ ]:
import re  # regular expressions for keyword search

PASS_THRESHOLD = 7  # minimum score to consider a response production-ready (out of 10)
# Defined here, adjacent to score_math_response(), so the threshold is always visible
# when reading the scoring function — no need to hunt through later cells.

def score_math_response(text):
    """
    Score a maths-tutor response on four dimensions (max 10 points).

    Parameters
    ----------
    text : str — the assistant's reply

    Returns
    -------
    tuple[int, bool] — (total_score, passed) where passed = score >= PASS_THRESHOLD
    """
    total = 0

    # ── Dimension 1: correct numeric answer present (3 pts) ──────────────────
    # re.search scans the WHOLE string (unlike re.match which anchors to the start)
    # \b = word boundary so '56' inside '156' does NOT falsely match
    has_answer = bool(re.search(r'\b56\b', text))
    pts = 3 if has_answer else 0
    icon = "✅" if has_answer else "❌"
    print(f"  {icon} Correct answer (56)   : {pts}/3")
    total += pts

    # ── Dimension 2: step-by-step explanation keywords (3 pts) ───────────────
    # Walk-through of the generator expression:
    #   re.search(pattern, text, re.I)  → Match object (truthy) or None (falsy)
    #   bool(...)                       → True or False
    #   int(bool(...))                  → 1 or 0  (converts boolean to countable integer)
    #   sum(... for w in step_words)    → total count of matched keywords
    # We use re.search (not re.match) so keywords anywhere in the string are found.
    # \b word boundaries prevent 'stepping' from matching 'step'.
    step_words = ["step", "first", "because", "multiply", "times"]
    step_hits = sum(
        int(bool(re.search(r'\b' + w + r'\b', text, re.I)))
        for w in step_words
    )  # int(bool(...)) converts Match→True→1 or None→False→0 for each word, then sums
    has_steps = step_hits >= 2   # at least 2 step-indicator words = structured explanation
    pts = 3 if has_steps else 0
    icon = "✅" if has_steps else "❌"
    print(f"  {icon} Step-by-step reasoning : {pts}/3 (hits={step_hits})")
    total += pts

    # ── Dimension 3: encouraging close (2 pts) ────────────────────────────────
    encourage_words = ["great", "well done", "keep", "proud", "excellent", "good job", "you can"]
    lower = text.lower()   # case-fold once; avoids re-lowering in the loop
    has_encourage = any(w in lower for w in encourage_words)
    pts = 2 if has_encourage else 0
    icon = "✅" if has_encourage else "❌"
    print(f"  {icon} Encouraging close      : {pts}/2")
    total += pts

    # ── Dimension 4: avoids giving direct answer (2 pts) ──────────────────
    # A good tutor withholds the solution and asks guiding questions instead.
    # re.search scans the whole string; \b boundaries prevent partial matches.
    # We check for x=1/5 and decimal equivalent x=0.2 as direct-answer signals.
    gives_answer = bool(re.search(
        r'\bx\s*=\s*1/5\b|\bx\s*=\s*0\.2\b|one.?fifth', text, re.I
    ))
    avoids_answer = not gives_answer   # tutor should NOT give the answer directly
    pts = 2 if avoids_answer else 0
    icon = "✅" if avoids_answer else "❌"
    print(f"  {icon} Avoids direct answer  : {pts}/2 (gives_answer={gives_answer})")
    total += pts

    passed = total >= PASS_THRESHOLD  # True if score meets the production bar
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"  {'─'*38}")
    print(f"  TOTAL SCORE            : {total}/10  →  {status} (threshold={PASS_THRESHOLD})")
    return total, passed

print("Scoring response WITHOUT system prompt:")
score_no_sys, _ = score_math_response(reply_no_sys)
print()
print("Scoring response WITH system prompt:")
score_with_sys, _ = score_math_response(reply_with_sys)

## CCA objective demonstrated: Multi-turn conversation — messages list accumulation and token cost analysis

## Section 6: Multi-Turn Conversation

In a multi-turn conversation the `messages` list grows with every exchange.
We call `client.messages.create()` **directly** here (not via `ask()`) so you can
see every `messages.append()` step explicitly.

**Architectural insight:** Every turn resends the full history — first-line clarity
is critical because the model re-reads all prior context on each call.

In [ ]:
messages = []  # start with an empty conversation history

# ── Turn 1 ────────────────────────────────────────────────────────────────────
# Append the first user message to the (currently empty) messages list
messages.append({"role": "user", "content": "What is photosynthesis?"})

# Call the API with the current messages list (1 entry)
r1 = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=150,
    messages=messages   # pass the growing list directly
)

# Append the assistant's reply so the next turn includes it as context
messages.append({"role": "assistant", "content": r1.content[0].text})

print("After turn 1, messages list has", len(messages), "entries")
print("Turn 1 input tokens:", r1.usage.input_tokens)
print()
print("Full messages list after turn 1:")
for i, m in enumerate(messages):  # print each entry so list growth is visible
    snippet = m["content"][:80].replace("\n", " ")
    print(f"  [{i}] role={m['role']!r:12s} content={snippet!r}")

# Record turn 1 in usage_log manually (direct create() calls bypass ask())
usage_log.append({
    "call": len(usage_log) + 1,
    "system": "",
    "prompt": "What is photosynthesis?",
    "input_tokens": r1.usage.input_tokens,
    "output_tokens": r1.usage.output_tokens,
    "stop_reason": r1.stop_reason,
})

print()
# ── Turn 2 ────────────────────────────────────────────────────────────────────
# The user asks a follow-up; the messages list now carries the prior exchange
messages.append({"role": "user", "content": "Give me a one-sentence summary."})

# The API receives ALL 3 entries (user, assistant, user) — full history is re-sent
r2 = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=80,
    messages=messages   # now contains 3 entries; entire history transmitted
)

messages.append({"role": "assistant", "content": r2.content[0].text})

print("After turn 2, messages list has", len(messages), "entries")
print("Turn 2 input tokens:", r2.usage.input_tokens, "← more because history is re-sent")
print("Token cost increase:", r2.usage.input_tokens - r1.usage.input_tokens)
print()
print("Full messages list after turn 2:")
for i, m in enumerate(messages):
    snippet = m["content"][:80].replace("\n", " ")
    print(f"  [{i}] role={m['role']!r:12s} content={snippet!r}")

# Record turn 2 in usage_log
usage_log.append({
    "call": len(usage_log) + 1,
    "system": "",
    "prompt": "Give me a one-sentence summary.",
    "input_tokens": r2.usage.input_tokens,
    "output_tokens": r2.usage.output_tokens,
    "stop_reason": r2.stop_reason,
})

print()
print("Architecture insight: turn-2 input_tokens > turn-1 because the model")
print("re-reads the entire conversation history on every call.")

## CCA objective demonstrated: Prompt Engineering — iterative prompt optimisation with a 3-version improvement cycle

## Section 7: Improvement Cycle — Write → Evaluate → Improve → Re-evaluate

We start from a weak prompt (V1), score it, identify gaps, revise to V2, score again,
and if still below threshold generate V3. The section ends with a side-by-side
comparison table of all three versions including a delta column.

> **Grader reliability note:** Deterministic rubrics can over-score responses that
> contain the right words but wrong semantics — for example, a response that uses
> the word "step" in an unrelated sentence would score 3/3 on Dimension 2 even if
> no actual step-by-step reasoning is present. For production evals, complement
> keyword checks with a Claude-as-judge semantic pass — the rule + judge layering
> pattern from the evaluation labs (see **Evaluation Lab 3: Claude-as-Judge**).
> The deterministic rubric here is fast and auditable; the semantic judge catches
> cases where the right tokens appear in the wrong reasoning structure.

In [ ]:
TASK = "Explain what 7 × 8 equals and why."

# ── Version 1: weak system prompt ─────────────────────────────────────────────
v1_system = "You are a tutor."
v1_reply  = ask(TASK, system=v1_system, max_tokens=200)
print("V1 reply snippet:", v1_reply[:120], "...\n")
print("V1 Rubric:")
v1_score, v1_passed = score_math_response(v1_reply)

# ── Version 2: improved system prompt ─────────────────────────────────────────
v2_system = (
    "You are a patient maths tutor for 10-year-old students. "
    "Always show your reasoning step by step using simple words. "
    "Keep your answer under 80 words. "
    "End with one encouraging sentence."
)
v2_reply  = ask(TASK, system=v2_system, max_tokens=200)
print("\nV2 reply snippet:", v2_reply[:120], "...\n")
print("V2 Rubric:")
v2_score, v2_passed = score_math_response(v2_reply)

# ── Version 3: generated conditionally if best < threshold ────────────────────
# max(v1_score, v2_score) tracks the BEST score achieved so far (not the latest),
# because a later version might regress — we want to know if ANY version cleared
# the bar, not just whether the most recent attempt did.
best_so_far = max(v1_score, v2_score)

if best_so_far < PASS_THRESHOLD:
    print(f"\nBest so far ({best_so_far}) < threshold ({PASS_THRESHOLD}) — iterating to V3...")
    v3_system = (
        "You are an expert maths tutor for 10-year-old students. "
        "Step 1: state the answer as '7 × 8 = 56'. "
        "Step 2: explain why using a simple real-world example (e.g. 7 groups of 8 apples). "
        "Keep the total response under 70 words. "
        "End with: 'You're doing great!'"
    )
    v3_reply = ask(TASK, system=v3_system, max_tokens=200)
    print("V3 reply snippet:", v3_reply[:120], "...\n")
    print("V3 Rubric:")
    v3_score, v3_passed = score_math_response(v3_reply)
else:
    # V2 already passed — generate V3 anyway for comparison richness
    print(f"\nV2 passed (score={v2_score}) — generating V3 for comparison.")
    v3_system = (
        "You are an expert maths tutor for 10-year-old students. "
        "Step 1: state the answer as '7 × 8 = 56'. "
        "Step 2: explain why using a simple real-world example (e.g. 7 groups of 8 apples). "
        "Keep the total response under 70 words. "
        "End with: 'You're doing great!'"
    )
    v3_reply = ask(TASK, system=v3_system, max_tokens=200)
    print("V3 reply snippet:", v3_reply[:120], "...\n")
    print("V3 Rubric:")
    v3_score, v3_passed = score_math_response(v3_reply)

# ── Side-by-side comparison table ─────────────────────────────────────────────
# max(v1_score, v2_score, v3_score): best across ALL versions (not just latest)
best_final = max(v1_score, v2_score, v3_score)

print()
print(f"{'Version':<8} {'Score':>6} {'Delta':>7}  {'System Prompt (truncated)'}")
print("─" * 72)

versions = [
    ("V1", v1_system, v1_score),
    ("V2", v2_system, v2_score),
    ("V3", v3_system, v3_score),
]
prev_score = None
for label, sys_text, sc in versions:
    delta_str = "—" if prev_score is None else f"{sc - prev_score:+d}"
    sys_snippet = sys_text[:45] + ".." if len(sys_text) > 45 else sys_text
    print(f"{label:<8} {sc:>5}/10 {delta_str:>7}  {sys_snippet}")
    prev_score = sc

print("─" * 72)
result_str = "✅ PASS" if best_final >= PASS_THRESHOLD else "❌ FAIL"
print(f"Best score: {best_final}/10  Threshold: {PASS_THRESHOLD}/10  →  {result_str}")

## CCA objective demonstrated: Agentic Architecture — anticipating, categorising, and handling API failure modes

## Section 8: Failure Mode Analysis

| Failure Mode | Trigger | Symptom | Root Cause | Fix / Guard |
|---|---|---|---|---|
| `system=None` passed explicitly | `system` key present with `None` value | `BadRequestError` / validation error | API schema requires `system` to be a string when key is present | Conditional: `if system is not None: kwargs['system'] = system` |
| `max_tokens` omitted entirely | `max_tokens` key absent from request | `ValidationError`: field required | `max_tokens` is a required parameter with no default | Always include `max_tokens`; set a sensible ceiling |
| Oversized `max_tokens` (> model max) | `max_tokens=999999` | `BadRequestError`: exceeds model limit | Each model has a fixed output-token ceiling | Check model docs; cap at `4096` for Haiku |
| Empty `messages` list | `messages=[]` | `BadRequestError`: messages must not be empty | API requires at least one turn | Validate `len(messages) > 0` before calling |
| Invalid model ID | `model="claude-fake-99"` | `NotFoundError` / `BadRequestError` | Model string does not match any deployed model | Use a constant for model name; catch `NotFoundError` |
| Non-alternating roles | Two consecutive `user` entries without `assistant` | `BadRequestError`: roles must alternate | API enforces user/assistant/user/... ordering | Ensure every user turn is followed by an assistant turn before the next user turn |

The live demo below deliberately triggers **four** of these and catches each one.
The remaining two (max_tokens omitted, oversized max_tokens) are described above
and can be triggered by removing `max_tokens` or setting it to `999999`.

In [ ]:
def trigger_and_catch(label, **kwargs):
    """Attempt an API call with broken kwargs; print the resulting error."""
    print(f"--- {label} ---")
    try:
        client.messages.create(**kwargs)
        print("  (no error — unexpected!)")  # should not reach here for broken calls
    except Exception as e:
        # Print the exception class name and a truncated message for readability
        print(f"  ERROR: {type(e).__name__}: {str(e)[:120]}")
    print()

# Failure 1: system=None passed as explicit keyword
trigger_and_catch(
    "system=None explicit",
    model="claude-haiku-4-5", max_tokens=50,
    system=None,
    messages=[{"role": "user", "content": "Hi"}],
)

# Failure 2: empty messages list
trigger_and_catch(
    "Empty messages list",
    model="claude-haiku-4-5", max_tokens=50,
    messages=[],
)

# Failure 3: invalid model ID
trigger_and_catch(
    "Invalid model ID",
    model="claude-fake-model-99", max_tokens=50,
    messages=[{"role": "user", "content": "Hi"}],
)

# Failure 4: non-alternating roles (user → user without assistant in between)
trigger_and_catch(
    "Non-alternating roles (user/user)",
    model="claude-haiku-4-5", max_tokens=50,
    messages=[
        {"role": "user", "content": "First message"},
        {"role": "user", "content": "Second message without assistant turn"},
    ],
)

print("All four failure modes caught cleanly — no unhandled exceptions.")

In [ ]:
# ── Output Variance Demo: the same vague prompt run 3 times ───────────────────
# This demonstrates a PROMPT-INDUCED failure mode (not an API error):
# non-deterministic output length/format breaks downstream parsers.

print("=== Output variance: same vague prompt × 3 ===")
variance_prompt = "Explain quantum computing."
word_counts = []   # collect word count of each reply

for i in range(3):   # run the same prompt three times
    reply = ask(variance_prompt, max_tokens=300)
    wc = len(reply.split())   # count whitespace-delimited words
    word_counts.append(wc)
    print(f"  Run {i+1}: {wc} words")

wc_min = min(word_counts)   # shortest response
wc_max = max(word_counts)   # longest response
wc_range = wc_max - wc_min  # spread across runs

print()
print(f"  Min words : {wc_min}")
print(f"  Max words : {wc_max}")
print(f"  Range     : {wc_range}")
print()
print("Production implication: This variance is a production bug —")
print("downstream parsers expecting consistent length or format will")
print("fail non-deterministically. Add explicit length constraints to")
print("your system prompt to tighten the distribution.")

## CCA objective demonstrated: Tracking per-call and cumulative token consumption with truncation detection

## Section 9: Token Usage Analysis

In [ ]:
# ── Per-call usage table with running cumulative totals ───────────────────────
# We accumulate cum_in and cum_out INSIDE the loop rather than summing post-hoc
# because this lets us print the running total on each row — essential for
# spotting which call consumed the most tokens at a glance.
# (Post-hoc sum would require a second pass and cannot produce per-row cumulative output.)

cum_in  = 0   # running total input tokens — incremented once per loop iteration
cum_out = 0   # running total output tokens — incremented once per loop iteration

header = f"{'Call':>4}  {'Input':>7}  {'Output':>7}  {'CumIn':>8}  {'CumOut':>8}  {'Stop':<12}  Prompt (truncated)"
print(header)
print("─" * len(header))

for entry in usage_log:
    # Accumulate: add this call's tokens to the running totals
    # Why outside a post-hoc sum? So each printed row shows the cumulative state
    # at that point in the session — useful for diagnosing cost spikes call-by-call.
    cum_in  += entry["input_tokens"]
    cum_out += entry["output_tokens"]
    prompt_snippet = entry["prompt"][:35]   # pre-extracted; avoids f-string nesting
    print(
        f"{entry['call']:>4}  "
        f"{entry['input_tokens']:>7}  "
        f"{entry['output_tokens']:>7}  "
        f"{cum_in:>8}  "
        f"{cum_out:>8}  "
        f"{entry['stop_reason']:<12}  "
        f"{prompt_snippet}"
    )

print("─" * len(header))
print(f"TOTAL  {cum_in:>7} input tokens   {cum_out:>7} output tokens")
print(f"Grand total tokens this session: {cum_in + cum_out}")
print()

# ── Truncation detection ──────────────────────────────────────────────────────
# If stop_reason == 'max_tokens' the model ran out of space — output is cut off.
# In production this is a bug: the caller receives an incomplete response.
print("Truncation check:")
truncated = [e for e in usage_log if e["stop_reason"] == "max_tokens"]
if truncated:
    for entry in truncated:
        print(f"  ⚠ Call {entry['call']} was TRUNCATED — increase max_tokens or shorten prompt")
else:
    print("  ✅ No truncated calls in this session.")
print()

# ── Output token variance: V1 vs V2 vs V3 prompt versions ────────────────────
# The improvement cycle made calls in sequence; they appear in usage_log by call number.
# Calls for V1/V2/V3 started after the system-prompt comparison calls (calls 1-2),
# multi-turn calls, then V1/V2/V3 — extract by finding entries with the task prompt.
task_snippet = TASK[:40]   # what was stored in usage_log for the improvement cycle
version_entries = [e for e in usage_log if e["prompt"].startswith(task_snippet[:20])]

if len(version_entries) >= 2:
    print("Output token variance across improvement cycle versions:")
    labels = ["V1", "V2", "V3"]
    for idx, entry in enumerate(version_entries[:3]):
        label = labels[idx] if idx < len(labels) else f"V{idx+1}"
        print(f"  {label}: {entry['output_tokens']} output tokens")
    if len(version_entries) >= 2:
        delta = version_entries[1]["output_tokens"] - version_entries[0]["output_tokens"]
        direction = "more" if delta > 0 else "fewer"
        print(f"  V2 used {abs(delta)} {direction} output tokens than V1.")
        print("  A more specific prompt typically reduces output tokens by constraining length.")
else:
    print("  (version_entries not found by prompt prefix — check usage_log manually)")
    for i, e in enumerate(usage_log):
        print(f"  Call {e['call']}: {e['output_tokens']} out tokens | prompt: {e['prompt'][:40]}")

## CCA objective demonstrated: Independent application of all lab concepts

## Section 10: Practice Drills

Complete the three exercises below. Each builds on a different section of this lab.

In [ ]:
# ── Drill 1 ───────────────────────────────────────────────────────────────────
# Write a system prompt that constrains Claude to answer ONLY questions about
# cooking. Call ask() with a cooking question AND a non-cooking question.
# Observe how the system prompt affects the off-topic response.

# YOUR CODE HERE
# cooking_system = "..."
# print(ask("How do I make pasta?", system=cooking_system))
# print(ask("What is the speed of light?", system=cooking_system))


# ── Drill 2 ───────────────────────────────────────────────────────────────────
# Extend score_math_response() with a 5th dimension: response must mention
# the word "multiplication" (2 bonus points). Re-score v1_reply and v2_reply.

# YOUR CODE HERE


# ── Drill 3 ───────────────────────────────────────────────────────────────────
# Build a 3-turn multi-turn conversation on a topic of your choice.
# After each turn, print the length of the messages list and the input_tokens used.
# Confirm that input tokens grow with each turn because history is re-sent.

# YOUR CODE HERE

print("Drills ready — add your code above each comment block.")

## CCA objective demonstrated: Exam-ready mental models synthesising all lab sections

## Section 11: CCA Takeaways

| # | Mental Model | One-liner |
|---|---|---|
| 1 | **system = developer's voice** | Anything that must persist invisibly across every turn belongs in `system`, not in `messages`. The API prepends it before the first user turn and it cannot be overridden. |
| 2 | **None ≠ absent** | Passing `system=None` is not the same as omitting `system`; guard with `if system is not None: kwargs['system'] = system`. |
| 3 | **Messages list = full history every turn** | Every API call re-sends all prior messages — input tokens grow with each turn; keep early messages concise. |
| 4 | **stop_reason as a signal** | `max_tokens` stop_reason means the response may be cut off — always check it in production with a truncation guard. |
| 5 | **Rubric before judge** | Deterministic keyword rubrics are fast but can be fooled by surface-level keyword matches; layer a Claude-as-judge semantic pass (Evaluation Lab 3) for production accuracy. |